In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

In [ ]:
MODEL_PATH = Path('./artifacts/5_beed_best_model.joblib')
INPUT_PATH = Path('./data/preprocessed/5_BEED_test.csv')
OUTPUT_PATH = Path('./data/preprocessed/5_beed_inference_predictions.csv')
LABEL = 'y'
CLASS_NAMES = {0: 'Healthy', 1: 'Generalized', 2: 'Focal', 3: 'Seizure Events'}

assert MODEL_PATH.exists(), f'Model not found: {MODEL_PATH.resolve()}'
assert INPUT_PATH.exists(), f'Input file not found: {INPUT_PATH.resolve()}'

bundle = joblib.load(MODEL_PATH)
if isinstance(bundle, dict):
    model = bundle.get('model', bundle.get('best_model', None))
    feature_columns = bundle.get('feature_columns', None)
else:
    model = bundle
    feature_columns = None

if model is None:
    raise ValueError('No model object found in artifact.')

In [ ]:
df = pd.read_csv(INPUT_PATH)
X = df.drop(columns=[LABEL], errors='ignore').copy()

if feature_columns is not None:
    missing = [c for c in feature_columns if c not in X.columns]
    if missing:
        raise ValueError(f'Missing required feature columns: {missing[:10]}')
    X = X.reindex(columns=feature_columns)

pred = model.predict(X)
pred_name = pd.Series(pred).map(CLASS_NAMES).fillna('Unknown')

out = pd.DataFrame({
    'prediction': pred,
    'prediction_name': pred_name
})

if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(X)
    for i in range(proba.shape[1]):
        out[f'prob_class_{i}'] = proba[:, i]

if LABEL in df.columns:
    out.insert(0, 'true_label', df[LABEL].values)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_PATH, index=False)
print(f'✅ Saved predictions to: {OUTPUT_PATH.resolve()}')
out.head()